# 03 - SOH Prediction

This notebook trains and evaluates SOH models using cross-battery generalization (train: B0005/B0006/B0007, test: B0018).

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load feature dataset

In [2]:
from src.data_loader import DEFAULT_BATTERY_IDS, download_battery_mat_files
from src.preprocessing import run_preprocessing_pipeline
from src.features import engineer_features_from_detailed

if not (PROCESSED_DIR / 'features_all.csv').exists():
    _ = download_battery_mat_files(RAW_DIR, DEFAULT_BATTERY_IDS)
    _, detailed_df = run_preprocessing_pipeline(RAW_DIR, PROCESSED_DIR, DEFAULT_BATTERY_IDS)
    feature_df = engineer_features_from_detailed(detailed_df)
    feature_df.to_csv(PROCESSED_DIR / 'features_all.csv', index=False)

feature_df = pd.read_csv(PROCESSED_DIR / 'features_all.csv')
feature_df.head()

,battery_id,cycle_number,capacity_Ah,avg_voltage,avg_current,avg_temperature,charge_time,discharge_time,energy_charged_Wh,SOH,...,max_temperature,temperature_rise,discharge_duration,energy_discharged,internal_resistance_proxy,capacity_fade_rate,cycle_efficiency,capacity_lag_1,capacity_lag_3,capacity_lag_5
0,B0005,1,1.862203,3.529829,1.818712,32.572328,7597.875,3690.234,3.276268,93.110144,...,38.982181,14.652147,3690.234,6.608778,0.216948,-0.004175,2.017167,1.862203,1.862203,1.862203
1,B0005,2,1.852078,3.537320,1.817644,32.725235,10516.000,3672.344,7.638814,92.603889,...,39.033398,14.335646,3672.344,6.586345,0.990131,-0.004175,0.862221,1.862203,1.862203,1.862203
2,B0005,3,1.841049,3.543737,1.816542,32.642862,10484.547,3651.641,7.608577,92.052437,...,38.818797,14.084531,3651.641,6.555683,26.274884,-0.004175,0.861618,1.852078,1.862203,1.862203
3,B0005,4,1.840912,3.543666,1.825618,32.514876,10397.890,3631.563,7.578063,92.045600,...,38.762305,14.108068,3631.563,6.554829,0.235616,-0.004175,0.864974,1.841049,1.862203,1.862203
4,B0005,5,1.840360,3.542343,1.826148,32.382349,10495.203,3629.172,7.565826,92.017996,...,38.665393,14.140596,3629.172,6.552232,0.094378,-0.004175,0.866030,1.840912,1.852078,1.862203


## Train SOH models

In [3]:
from src.models import run_soh_benchmark

soh_metrics, soh_results = run_soh_benchmark(
    feature_df=feature_df,
    models_dir=MODELS_DIR,
    test_battery='B0018',
    sequence_length=10,
)
soh_metrics_df = pd.DataFrame(soh_metrics).sort_values('RMSE').reset_index(drop=True)
soh_metrics_df

,Model,Task,RMSE,MAE,MAPE,R2,Train_Time_s
0,Linear Regression,SOH,0.116027,0.099809,0.127294,0.999769,0.000410
1,XGBoost Regressor,SOH,0.565523,0.367674,0.472972,0.994523,4.127144
2,Random Forest Regressor,SOH,0.714788,0.408414,0.530208,0.991250,0.169846
3,Ridge Regression,SOH,0.943786,0.885303,1.143149,0.984746,0.000331
4,CNN-BiLSTM,SOH,2.168477,1.769527,2.234558,0.902411,1.866639
5,LSTM Neural Network,SOH,6.947546,6.265347,8.098037,-0.001736,1.083232


## Plot actual vs predicted SOH curves

In [4]:
import numpy as np
from src.visualisation import plot_actual_vs_predicted

for model_name, payload in soh_results.items():
    safe_name = model_name.lower().replace(' ', '_').replace('-', '_')
    plot_actual_vs_predicted(
        cycle_number=np.asarray(payload['cycle_number']),
        y_true=np.asarray(payload['y_true']),
        y_pred=np.asarray(payload['predictions']),
        title=f'SOH: Actual vs Predicted ({model_name}) on B0018',
        y_label='SOH (%)',
        output_path=FIG_DIR / f'soh_actual_vs_pred_{safe_name}.png',
    )

print('Saved SOH prediction plots to results/figures/.')

Saved SOH prediction plots to results/figures/.
